# Phase 4 -- Trial-count-matched SMI comparison (saline vs. randomly-subsampled DCZ trials)

Complementary to `4.SessionComparison_TrialMatched.ipynb`'s sequential-block approach, not a replacement -- that notebook answers "does the DREADD effect have a time course within the DCZ session" (blocks are contiguous, in temporal order). This one answers the original question directly: **is the saline-vs-DCZ difference just an artifact of DCZ having more trials?** Repeatedly draw a random subset of `N` DCZ trials (`N` = saline's own trial count, no replacement), recompute SMI + reliability on each draw, and aggregate across draws -- an unbiased, low-variance estimate of "what would this DCZ session look like at N trials," averaging over the full space of possible subsets rather than looking at one temporal slice at a time.

**Aggregation**: `median_SMI` per cell across draws where that cell was individually `valid` (reported alongside `mean_SMI` for comparison, consistent with how this pipeline treats SMI as bounded/non-normal elsewhere -- median is primary). A cell counts as `valid` in the final aggregate if it was valid in at least `VALID_FRACTION_THRESHOLD` (default 0.5, i.e. a majority) of the `N_REPEATS` draws -- both constants below, easy to change.

**`N_REPEATS` is a real cost/precision tradeoff, worth understanding before running this.** Each repeat re-runs the full `combined_reliability_test_improved` + `calculate_SMI_improved` pipeline on ~500-700 cells -- the same expensive computation `4.SessionComparison_TrialMatched.ipynb`'s per-block runs already took real time to complete. `N_REPEATS=20` here is a deliberately modest default (an order of magnitude more stable than a single draw, without multiplying runtime into the "many hours" range `N_REPEATS=100+` would reach across 5 groups x 2 animals) -- turn it up if you want a more precise aggregate and are fine waiting longer.

**Response plots**: included, but only as an illustrative single example -- the actual comparison numbers (SMI, reliable/valid fractions) come from the full aggregate across all `N_REPEATS` draws, not from any one draw, so there's no single "the" DCZ response profile to show the way each block had one in the sequential-block notebook. The response plot here uses just the *first* draw's trial subset, clearly labeled as one example, not the aggregate.

**Starts from `preproc.h5` directly**, same as the sequential-block notebook -- not from Phase 3's saved `*_smi_results_dreadd.h5` (fixed at full trial count). Layer identity (`find_layer_curve_path`/`load_session_layer_cells_for_smi`, Phase 1's output) is loaded once per session from the start this time, and the saved per-cell table matches the same `cell_idx/SMI/valid/analysis_reliable/layer/condition` shape as the sequential-block notebook's -- **saved under a different filename** (`{group}_random_subsampled_comparison_table.csv`, different output folder) so the two methods' results never collide or overwrite each other.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob
from collections import Counter

import numpy as np
import pandas as pd
import h5py
import matplotlib
matplotlib.use('Qt5Agg')  # for plt.show() popups
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Existing pipeline code, reused via import -- not modified.
from helper import files
from helper.SMI_Calculation import calculate_SMI_improved
from helper.ReliabilityTesting import combined_reliability_test_improved
from helper.SMICalculation_LayerSpecific_SingleRecording import filter_onset_response_cells
from helper.ResponseVisualization import create_response_plot

N_SHUFFLES = 100  # combined_reliability_test_improved's own internal shuffle count (unrelated to N_REPEATS below) --
                  # lowered from 300: aggregate_repeated_subsamples_by_averaged_evidence (see below) averages
                  # avg_cc/cohen_d across the N_REPEATS draws, so each individual repeat's shuffle-null estimate
                  # doesn't need to be as precise -- trades a little per-repeat precision for ~3x less runtime
                  # across N_REPEATS x n_pairs x n_animals
N_REPEATS = 20  # how many random trial-subsamples to draw and aggregate over -- see markdown above for the cost/precision tradeoff
VALID_FRACTION_THRESHOLD = 0.5  # a cell counts as "valid" in the aggregate if valid in >= this fraction of repeats
RANDOM_SUBSAMPLE_BASE_SEED = 0  # fixed seed -- reruns are reproducible, not a fresh random draw each time

# V1 (non-RSC) reliability parameters -- same branch Preprocess.py itself
# uses for this project's recordings.
RELIABILITY_KWARGS = dict(
    cc_percentile=90, cohen_threshold=0.8, min_cc_threshold=0.1,
    min_pattern_corr=0.3, peak_distance_threshold=5,
    use_activity_threshold=True, activity_method='absolute_percentile',
    # pattern_test='shuffled' + pattern_percentile/peak_distance_percentile=80 --
    # real-data diagnostics on JSY090 DCZ_1 found the DEFAULT 'fixed' pattern test
    # (min_pattern_corr/peak_distance_threshold as flat floors) was the actual
    # bottleneck behind the random-subsample method's low reliable-cell counts,
    # not the correlation test (which passed ~92-97% of full-session-reliable
    # cells on its own). Specifically peak_distance_threshold=5 bins: at only
    # ~10 trials/half, a genuinely reliable cell's mean peak distance across
    # repeats was ~19 bins (vs. the full session's precise estimate), because
    # argmax is a low-information, high-variance statistic at this trial count
    # -- no fixed OR shuffle-relative threshold rescues that if set too strict
    # (pattern_test='shuffled' at the default 95th percentile made things
    # *worse*: 122 full-session-reliable cells found instead of 152, and vote
    # recovery dropped from 24.3% to 15.6%). A percentile sweep (95/85/80/70/none)
    # showed a smooth, monotonic precision/recall tradeoff -- 80 was chosen as
    # the sweet spot: recovers ~60% more real cells than the old flat-floor
    # default (60 vs 37 of 152 full-session-reliable cells) while keeping
    # precision reasonably high (81.1% vs 97.4%), clearly better than 70th
    # percentile's tradeoff (70.1% precision) or dropping the gate entirely
    # (50.8% precision). See conversation history for the full sweep and the
    # false-positive check on newly-recovered cells specifically.
    pattern_test='shuffled', pattern_percentile=80, peak_distance_percentile=80,
)

# Same defaults run_smi_analysis_session (3.SMICalculation.py) currently uses.
SMI_KWARGS = dict(
    exclude_first_bins=10, exclude_last_bins=10,
    segment_distance=28, exclude_start_cm=15, exclude_end_cm=10,
    smoothing_sigma=1.0,
)

# Point this at whichever animal you're processing -- kept for both so
# switching doesn't leave other TEST_* constants pointing at the wrong
# animal (same fix 4.SessionComparison_TrialMatched.ipynb needed).
TEST_ANIMAL_DIR_JSY090 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"
TEST_ANIMAL_DIR_JSY093 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"
TEST_ANIMAL_DIR = TEST_ANIMAL_DIR_JSY090

## Setup -- `discover_smi_sessions` / `pair_saline_dcz_sessions`

Reused unchanged from `4.SessionComparison_TrialMatched.ipynb`.

In [2]:
def discover_smi_sessions(animal_dir):
    """
    Scan animal_dir for every already-computed *_smi_results_dreadd.h5
    file, labeling each by whichever known naming pattern its TSeries
    folder matches.

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'save_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'),
                                   recursive=True))

    entries = []
    unmatched = []

    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, save_path, tseries_dir, tseries_name))

    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, save_path, tseries_dir, tseries_name in entries:
        label = f'{base_label}__{tseries_name}' if label_counts[base_label] > 1 else base_label

        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['save_path']}, skipping {save_path}")
            continue

        catalog[label] = {'save_path': save_path, 'session_type': session_type, 'tseries_dir': tseries_dir}

    print(f"Discovered {len(catalog)} sessions with saved SMI results under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['save_path']}")

    collided_labels = [l for l, c in label_counts.items() if c > 1]
    if collided_labels:
        print(f"\n{len(collided_labels)} label(s) had multiple TSeries and were disambiguated: {collided_labels}")
    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern (labeled 'unknown'): {unmatched}")

    return catalog


def pair_saline_dcz_sessions(session_catalog):
    """
    Pair each saline session with its same-day dcz session, by shared
    parent folder -- robust to whatever label string discover_smi_sessions
    assigned.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.

    Returns
    -------
    pairs : list of dict
        Each: {'group_name', 'saline_label', 'dcz_label'}.
    """
    saline_entries = [(l, info) for l, info in session_catalog.items() if info['session_type'] == 'saline']
    dcz_entries = [(l, info) for l, info in session_catalog.items() if info['session_type'] == 'dcz']

    pairs = []
    for saline_label, saline_info in saline_entries:
        saline_parent = os.path.dirname(saline_info['tseries_dir'])
        matches = [(l, info) for l, info in dcz_entries
                   if os.path.dirname(info['tseries_dir']) == saline_parent]
        if len(matches) != 1:
            print(f"WARNING: saline session '{saline_label}' has {len(matches)} same-parent-folder dcz "
                  f"match(es) (expected 1) -- skipping. Parent: {saline_parent}")
            continue
        dcz_label, dcz_info = matches[0]
        pairs.append({
            'group_name': os.path.basename(saline_parent),
            'saline_label': saline_label,
            'dcz_label': dcz_label,
        })

    print(f"Paired {len(pairs)} saline/dcz session(s): {[p['group_name'] for p in pairs]}")
    return pairs

In [ ]:
# --- Try it on the real animal dir ---
smi_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)
session_pairs = pair_saline_dcz_sessions(smi_catalog)
for p in session_pairs:
    print(p)

## Setup -- save utilities

In [3]:
def save_dataframe_csv(df, output_dir, filename):
    """Save a DataFrame to {output_dir}/{filename}, creating output_dir if needed."""
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=False)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """Save a matplotlib figure to {output_dir}/{filename}, creating output_dir if needed."""
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path

## Setup -- layer assignment

Reused unchanged from `4.SessionComparison_TrialMatched.ipynb` (itself reimplemented unchanged from `3.SMICalculation.py`) -- built in from the start this time rather than added as a follow-up.

In [4]:
def find_layer_curve_path(plane0_path, prefer_averaged=True):
    """Locate a session's Phase 1 layer-curve-results file, given its suite2p/plane0 path."""
    tseries_dir = os.path.dirname(os.path.dirname(str(plane0_path)))
    averaged_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results_averaged.h5'))
    independent_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results.h5'))

    if prefer_averaged and averaged_matches:
        matches = averaged_matches
    elif independent_matches:
        matches = independent_matches
    elif averaged_matches:
        matches = averaged_matches
    else:
        raise FileNotFoundError(f"No *_layer_curve_results(_averaged).h5 found in {tseries_dir} "
                                 "-- has Phase 1 been run for this session?")

    if len(matches) > 1:
        print(f"WARNING: multiple matches in {tseries_dir}, using {matches[0]}")
    return matches[0]


def load_session_layer_cells_for_smi(plane0_path, layer_names=('L2/3', 'L4', 'L5', 'L6')):
    """
    Load Phase 1's curve-based layer assignment for one session, converted
    to {layer_name: indices}.
    """
    layer_curve_path = find_layer_curve_path(plane0_path)

    with h5py.File(layer_curve_path, 'r') as f:
        layer_codes = f['layer_codes'][:]
        saved_layer_names = tuple(n.decode() if isinstance(n, bytes) else n
                                   for n in f['layer_names'][:])

    if saved_layer_names != layer_names:
        print(f"NOTE: saved layer_names {saved_layer_names} differ in order from "
              f"the requested {layer_names} -- using the saved order.")

    layer_cells = {name: np.where(layer_codes == code)[0] for code, name in enumerate(saved_layer_names)}

    print(f"Loaded layer assignment from {layer_curve_path}")
    for name, idx in layer_cells.items():
        print(f"  {name}: {len(idx)} cells")

    return layer_cells


def load_session_layer_of_cell(tseries_dir, n_cells):
    """
    Per-cell layer label array for one session.

    Parameters
    ----------
    tseries_dir : str
    n_cells : int

    Returns
    -------
    layer_of_cell : numpy.ndarray of object, shape (n_cells,)
    """
    plane0_path = os.path.join(tseries_dir, 'suite2p', 'plane0')
    layer_cells = load_session_layer_cells_for_smi(plane0_path)

    layer_of_cell = np.full(n_cells, None, dtype=object)
    for layer_name, idx in layer_cells.items():
        layer_of_cell[idx] = layer_name
    return layer_of_cell

## Function -- `load_session_spatial_data`

Reused unchanged from `4.SessionComparison_TrialMatched.ipynb`.

In [5]:
def load_session_spatial_data(tseries_dir):
    """
    Pull spatial_activity/norm_spatial_activity/bin_centers straight from
    one session's preproc.h5.

    Parameters
    ----------
    tseries_dir : str

    Returns
    -------
    spatial_activity, norm_spatial_activity : numpy.ndarray
        (n_cells, n_trials, n_bins).
    bin_centers : numpy.ndarray
        (n_bins,).
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    preproc_data = files.read_h5(preproc_files[0])
    spatial_activity = preproc_data['spatial_activity']
    norm_spatial_activity = preproc_data['norm_spatial_activity']
    bin_centers = preproc_data['bin_centers']
    print(f"Loaded spatial_activity {spatial_activity.shape} from {tseries_dir}")
    return spatial_activity, norm_spatial_activity, bin_centers

In [ ]:
# --- Try it on the real first saline/dcz sessions ---
test_pair = session_pairs[0]
test_saline_activity, test_saline_norm, test_saline_bins = load_session_spatial_data(
    smi_catalog[test_pair['saline_label']]['tseries_dir']
)
test_dcz_activity, test_dcz_norm, test_dcz_bins = load_session_spatial_data(
    smi_catalog[test_pair['dcz_label']]['tseries_dir']
)
print(f"\nsaline trials: {test_saline_activity.shape[1]}   dcz trials: {test_dcz_activity.shape[1]}")

## Function -- `run_smi_and_reliability_for_trial_subset`

Same core computation as `4.SessionComparison_TrialMatched.ipynb`'s `run_smi_and_reliability_for_block` -- `combined_reliability_test_improved` (V1 parameters) -> `filter_onset_response_cells` -> `calculate_SMI_improved`, all unmodified -- but takes an arbitrary array of trial **indices** instead of a contiguous `(start, end)` range, so it works for both a sequential block (`np.arange(start, end)`) and a random subsample (`rng.choice(...)`, no replacement).

In [ ]:
def run_smi_and_reliability_for_trial_subset(spatial_activity, bin_centers, trial_indices,
                                              n_shuffles=N_SHUFFLES,
                                              reliability_kwargs=RELIABILITY_KWARGS,
                                              smi_kwargs=SMI_KWARGS):
    """
    Recompute SMI + reliability from scratch on an arbitrary trial-index
    subset of spatial_activity (not necessarily contiguous).

    Parameters
    ----------
    spatial_activity : numpy.ndarray
        (n_cells, n_trials, n_bins) -- the FULL session's array;
        trial_indices selects which trials on its trial axis.
    bin_centers : numpy.ndarray
        Raw cm-scale bin centers for this session.
    trial_indices : numpy.ndarray of int
        Which trials to use, in any order (no replacement is not enforced
        here -- the caller decides how trial_indices was chosen).
    n_shuffles : int
        Passed to combined_reliability_test_improved.
    reliability_kwargs : dict
    smi_kwargs : dict
        exclude_first_bins/exclude_last_bins go to filter_onset_response_cells;
        segment_distance/exclude_start_cm/exclude_end_cm/smoothing_sigma go
        to calculate_SMI_improved.

    Returns
    -------
    result : dict
        {'n_trials', 'trial_indices', 'combined_reliable', 'pattern_reliable',
        'active_cells', 'analysis_reliable_cells', 'valid_cells_mask',
        'SMI', 'avg_cc', 'cohen_d', 'peak_distances'}. 'peak_distances' is
        propagated (previously computed internally but discarded) so
        aggregate_repeated_subsamples_by_averaged_evidence below can
        average it across repeats along with avg_cc/cohen_d.
    """
    subset_activity = spatial_activity[:, trial_indices, :]
    n_trials = subset_activity.shape[1]

    shifted_centers = bin_centers - np.min(bin_centers)
    scaled_bin_centers = shifted_centers * (np.size(bin_centers) / np.max(shifted_centers))

    (combined_reliable, reliable_cells, pattern_reliable, avg_cc, cohens_d,
     odd_even_corr, peak_distances, active_cells) = combined_reliability_test_improved(
        subset_activity, n_shuffles=n_shuffles, **reliability_kwargs
    )

    non_onset_cells, rejected_info = filter_onset_response_cells(
        subset_activity, scaled_bin_centers, combined_reliable,
        exclude_first_bins=smi_kwargs['exclude_first_bins'],
        exclude_last_bins=smi_kwargs['exclude_last_bins'],
    )
    analysis_reliable_cells = combined_reliable & non_onset_cells

    smi_results = calculate_SMI_improved(
        subset_activity, scaled_bin_centers, analysis_reliable_cells,
        segment_distance=smi_kwargs['segment_distance'],
        exclude_start_cm=smi_kwargs['exclude_start_cm'],
        exclude_end_cm=smi_kwargs['exclude_end_cm'],
        smoothing_sigma=smi_kwargs['smoothing_sigma'],
    )

    print(f"  {n_trials} trials: combined_reliable={int(combined_reliable.sum())}, "
          f"analysis_reliable={int(analysis_reliable_cells.sum())}, "
          f"valid={int(smi_results['reliable_valid_cells'].sum())}")

    return {
        'n_trials': n_trials,
        'trial_indices': trial_indices,
        'combined_reliable': combined_reliable,
        'pattern_reliable': pattern_reliable,
        'active_cells': active_cells,
        'analysis_reliable_cells': analysis_reliable_cells,
        'valid_cells_mask': smi_results['reliable_valid_cells'],
        'SMI': smi_results['SMI'],
        'avg_cc': avg_cc,
        'cohen_d': cohens_d,
        'peak_distances': peak_distances,
    }

In [ ]:
# --- Sanity check: running this on saline's OWN full trial set (all indices,
#     in order) should reproduce a whole-session result, same as the
#     sequential-block notebook's single implicit saline block ---
test_saline_result = run_smi_and_reliability_for_trial_subset(
    test_saline_activity, test_saline_bins, np.arange(test_saline_activity.shape[1])
)

## Function -- `repeated_random_subsample_smi`

Draws `n_repeats` independent random subsets of `n_trials_target` trials (no replacement) from the DCZ session, calling `run_smi_and_reliability_for_trial_subset` on each. A fixed base seed makes this reproducible across reruns rather than genuinely random every time.

In [7]:
def repeated_random_subsample_smi(dcz_activity, dcz_bins, n_trials_target, n_repeats=N_REPEATS,
                                   base_seed=RANDOM_SUBSAMPLE_BASE_SEED, n_shuffles=N_SHUFFLES):
    """
    Repeatedly draw a random n_trials_target-sized subset of dcz_activity's
    trials (no replacement) and recompute SMI + reliability on each draw.

    Parameters
    ----------
    dcz_activity : numpy.ndarray
        (n_cells, n_trials, n_bins) -- the dcz session's full array.
    dcz_bins : numpy.ndarray
    n_trials_target : int
        Trials per draw -- the paired saline session's own trial count.
    n_repeats : int
    base_seed : int
        rng = np.random.default_rng(base_seed + repeat_idx) per repeat --
        fixed and reproducible, not fresh-random every call.
    n_shuffles : int
        Passed through to run_smi_and_reliability_for_trial_subset.

    Returns
    -------
    repeat_results : list of dict
        One run_smi_and_reliability_for_trial_subset(...) result per repeat,
        in draw order.
    """
    n_trials_dcz = dcz_activity.shape[1]
    repeat_results = []
    for repeat_idx in range(n_repeats):
        rng = np.random.default_rng(base_seed + repeat_idx)
        trial_indices = rng.choice(n_trials_dcz, size=n_trials_target, replace=False)
        trial_indices.sort()  # cosmetic only -- doesn't affect the trial-averaged computation below
        print(f"--- Repeat {repeat_idx + 1}/{n_repeats} ---")
        result = run_smi_and_reliability_for_trial_subset(
            dcz_activity, dcz_bins, trial_indices, n_shuffles=n_shuffles
        )
        repeat_results.append(result)

    return repeat_results

In [ ]:
# --- Quick real-data check with a small n_repeats (fast) -- the real run
#     uses N_REPEATS (20 by default), this is just to confirm the loop works ---
test_repeat_results = repeated_random_subsample_smi(
    test_dcz_activity, test_dcz_bins, test_saline_activity.shape[1], n_repeats=3
)
print(f"\nGot {len(test_repeat_results)} repeat result(s).")

## Function -- `aggregate_repeated_subsamples`

Turns a list of per-repeat results into one per-cell summary: `median_SMI`/`mean_SMI` (computed only over the repeats where that specific cell was `valid` -- a cell that fails validity in some draws just by chance of which trials got picked shouldn't have those draws' garbage SMI values dragging its average down), `valid_fraction`/`reliable_fraction` (how often across repeats), and a final `valid` call (`valid_fraction >= VALID_FRACTION_THRESHOLD`).

In [8]:
def aggregate_repeated_subsamples(repeat_results, valid_fraction_threshold=VALID_FRACTION_THRESHOLD):
    """
    Aggregate repeated_random_subsample_smi's per-repeat results into one
    per-cell summary.

    Parameters
    ----------
    repeat_results : list of dict
        From repeated_random_subsample_smi.
    valid_fraction_threshold : float

    Returns
    -------
    aggregated : dict
        {'n_repeats', 'median_SMI', 'mean_SMI', 'valid_fraction',
        'reliable_fraction', 'n_valid_repeats', 'valid_cells_mask',
        'analysis_reliable_cells'} -- each array-valued entry has shape
        (n_cells,). 'valid_cells_mask'/'analysis_reliable_cells' here are
        the FINAL aggregated calls (valid_fraction/reliable_fraction >=
        threshold), for compatibility with the same downstream
        table-building/plotting functions the sequential-block notebook
        uses.
    """
    n_repeats = len(repeat_results)
    n_cells = len(repeat_results[0]['SMI'])

    valid_stack = np.stack([r['valid_cells_mask'] for r in repeat_results])          # (n_repeats, n_cells)
    reliable_stack = np.stack([r['analysis_reliable_cells'] for r in repeat_results])  # (n_repeats, n_cells)
    smi_stack = np.stack([r['SMI'] for r in repeat_results])                          # (n_repeats, n_cells)

    valid_fraction = valid_stack.mean(axis=0)
    reliable_fraction = reliable_stack.mean(axis=0)
    n_valid_repeats = valid_stack.sum(axis=0)

    median_SMI = np.full(n_cells, np.nan)
    mean_SMI = np.full(n_cells, np.nan)
    for cell in range(n_cells):
        cell_valid_smi = smi_stack[valid_stack[:, cell], cell]
        if len(cell_valid_smi) > 0:
            median_SMI[cell] = np.median(cell_valid_smi)
            mean_SMI[cell] = np.mean(cell_valid_smi)

    final_valid = valid_fraction >= valid_fraction_threshold
    final_reliable = reliable_fraction >= valid_fraction_threshold

    print(f"Aggregated {n_repeats} repeats: "
          f"{int(final_valid.sum())}/{n_cells} cells valid in >={valid_fraction_threshold:.0%} of repeats, "
          f"{int(final_reliable.sum())}/{n_cells} reliable in >={valid_fraction_threshold:.0%}")

    return {
        'n_repeats': n_repeats,
        'median_SMI': median_SMI,
        'mean_SMI': mean_SMI,
        'valid_fraction': valid_fraction,
        'reliable_fraction': reliable_fraction,
        'n_valid_repeats': n_valid_repeats,
        'valid_cells_mask': final_valid,
        'analysis_reliable_cells': final_reliable,
    }

## Function -- `aggregate_repeated_subsamples_by_averaged_evidence`

An alternative to `aggregate_repeated_subsamples`'s `analysis_reliable_cells`/`reliable_fraction`-vote call (kept alongside it, not replacing it, so both can be compared on real data before picking one). Real-data diagnostic on JSY090 DCZ_1 found the vote-based aggregate (`reliable_fraction >= 0.5`) recovers only 38 cells overall -- *fewer* than the ~60 an average single repeat already finds on its own, and only 24.3% of the cells known-reliable on DCZ's full session, because `combined_reliability_test_improved`'s correlation/pattern tests are one-sided threshold tests: at this reduced trial count, a genuinely reliable cell's per-repeat pass probability is often well under 50% (median 0.15 among cells reliable on the full session) -- voting on 20 already-thresholded pass/fail outcomes throws away exactly the "how close to threshold" information that would separate a consistently-marginal-but-real cell from a consistently-not-reliable one.

This function instead averages the **continuous** per-repeat evidence (`avg_cc`, `cohen_d`, `peak_distances` -- all already returned by `run_smi_and_reliability_for_trial_subset`, no helper-function changes needed) across the `N_REPEATS` draws first, then applies `RELIABILITY_KWARGS`'s existing fixed cutoffs **once** to the averaged, much-less-noisy values -- reducing sampling noise in the estimate itself rather than voting on noisy binary decisions built from it.

One simplification vs. the original per-repeat decision, flagged explicitly: the original correlation criterion also requires `avg_cc` to beat that specific repeat's own shuffle-null 90th-percentile threshold -- that per-repeat threshold value isn't exposed by `combined_reliability_test_improved`'s return signature, so it isn't reproduced here. `cohen_d` (already a shuffle-normalized effect size -- real cc vs. that repeat's null mean and spread) is still averaged and thresholded, so the "beat the null" intent is still represented, just via Cohen's d rather than the raw percentile comparison.</cell id="900a2015">


In [ ]:
def aggregate_repeated_subsamples_by_averaged_evidence(repeat_results, reliability_kwargs=RELIABILITY_KWARGS,
                                                        activity_fraction_threshold=VALID_FRACTION_THRESHOLD):
    """
    Alternative to aggregate_repeated_subsamples's reliability call: average
    the continuous avg_cc/cohen_d/peak_distances across repeats first, then
    apply reliability_kwargs's fixed cutoffs once -- see markdown above for
    the real-data motivation and the one simplification made (no per-repeat
    shuffle-threshold comparison, since that value isn't exposed by
    combined_reliability_test_improved).

    avg_cc/cohen_d are only averaged over the repeats where that cell passed
    the activity-threshold screen that repeat (same reasoning as
    aggregate_repeated_subsamples's median_SMI -- a repeat that skipped the
    cell entirely shouldn't drag its average toward the default-zero value
    test_cell_reliability_improved leaves in place for skipped cells).
    peak_distances is averaged over all repeats (the pattern test doesn't
    skip low-activity cells the way the correlation test does). 'active' is
    left as a majority vote across repeats -- unlike the correlation/pattern
    tests, the activity threshold isn't rebuilding a fresh noisy null each
    repeat, so it isn't the source of the problem this function addresses.

    Parameters
    ----------
    repeat_results : list of dict
        From repeated_random_subsample_smi (needs 'avg_cc', 'cohen_d',
        'peak_distances', 'active_cells' per repeat).
    reliability_kwargs : dict
        Reuses min_cc_threshold/cohen_threshold/min_pattern_corr/
        peak_distance_threshold -- same nominal criteria as the per-repeat
        test, applied once instead of voted on 20 times.
    activity_fraction_threshold : float
        Fraction of repeats a cell must be 'active' in to count as active
        in the aggregate.

    Returns
    -------
    aggregated : dict
        {'avg_cc_mean', 'cohen_d_mean', 'peak_distance_mean',
        'active_fraction', 'n_active_repeats', 'analysis_reliable_cells'} --
        each array-valued entry has shape (n_cells,).
        'analysis_reliable_cells' here is this function's own re-derived
        reliability call (averaged-evidence, single threshold) -- combine
        with aggregate_repeated_subsamples's median_SMI/valid_cells_mask
        (SMI value + positional-validity are unaffected by which
        reliability aggregation is used) to get a full result, the way
        run_trial_matched_random_subsample_comparison_v2 below does.
    """
    n_repeats = len(repeat_results)
    n_cells = len(repeat_results[0]['avg_cc'])

    active_stack = np.stack([r['active_cells'] for r in repeat_results])        # (n_repeats, n_cells)
    avg_cc_stack = np.stack([r['avg_cc'] for r in repeat_results])
    cohen_d_stack = np.stack([r['cohen_d'] for r in repeat_results])
    peak_dist_stack = np.stack([r['peak_distances'] for r in repeat_results])

    active_fraction = active_stack.mean(axis=0)
    n_active_repeats = active_stack.sum(axis=0)

    avg_cc_mean = np.full(n_cells, np.nan)
    cohen_d_mean = np.full(n_cells, np.nan)
    for cell in range(n_cells):
        active_mask = active_stack[:, cell]
        if active_mask.any():
            avg_cc_mean[cell] = avg_cc_stack[active_mask, cell].mean()
            cohen_d_mean[cell] = cohen_d_stack[active_mask, cell].mean()

    # pattern test doesn't skip cells for low activity -- safe to average over all repeats
    peak_distance_mean = peak_dist_stack.mean(axis=0)

    active_in_aggregate = active_fraction >= activity_fraction_threshold
    reliable_by_cc = np.nan_to_num(avg_cc_mean, nan=-np.inf) > reliability_kwargs['min_cc_threshold']
    reliable_by_cohen = np.nan_to_num(cohen_d_mean, nan=-np.inf) > reliability_kwargs['cohen_threshold']
    pattern_reliable_agg = ((np.nan_to_num(avg_cc_mean, nan=-np.inf) >= reliability_kwargs['min_pattern_corr'])
                            & (peak_distance_mean <= reliability_kwargs['peak_distance_threshold']))

    combined_reliable_agg = active_in_aggregate & reliable_by_cc & reliable_by_cohen & pattern_reliable_agg

    print(f"Averaged-evidence aggregate ({n_repeats} repeats): "
          f"{int(active_in_aggregate.sum())}/{n_cells} active (>={activity_fraction_threshold:.0%} of repeats), "
          f"{int(combined_reliable_agg.sum())}/{n_cells} reliable (averaged cc/cohen_d/pattern, single threshold)")

    return {
        'avg_cc_mean': avg_cc_mean,
        'cohen_d_mean': cohen_d_mean,
        'peak_distance_mean': peak_distance_mean,
        'active_fraction': active_fraction,
        'n_active_repeats': n_active_repeats,
        'analysis_reliable_cells': combined_reliable_agg,
    }

In [ ]:
# --- Try it on the real (small n_repeats) test results above ---
test_aggregated = aggregate_repeated_subsamples(test_repeat_results)
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in test_aggregated.items()})

## Function -- `run_trial_matched_random_subsample_comparison`

Combines everything above for one saline/dcz pair: saline computed once (its own full trial set), dcz computed via `N_REPEATS` random draws + aggregation. Returns the same `block_results`/`session_data` shape the sequential-block notebook's downstream functions expect (`'saline'` and one other condition -- named `'dcz_random_subsampled'` here instead of `'dcz_block_N'`), so `build_trial_matched_comparison_table` and `plot_smi_across_blocks` can be reused completely unchanged.

The response-plot step uses the *first* repeat's actual trial subset as an illustrative single example (see title markdown) -- not the aggregate, since there's no single trial subset that "is" the aggregate.

In [9]:
def run_trial_matched_random_subsample_comparison(session_catalog, saline_label, dcz_label,
                                                    n_repeats=N_REPEATS, n_shuffles=N_SHUFFLES):
    """
    Loads one saline/dcz pair, computes saline once (full trial set) and
    dcz via repeated random subsampling + aggregation.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    saline_label, dcz_label : str
    n_repeats, n_shuffles : int

    Returns
    -------
    block_results : dict
        {'saline': ..., 'dcz_random_subsampled': ...} -- same shape as
        run_smi_and_reliability_for_trial_subset's own return value, plus
        'n_repeats' on the dcz entry.
    session_data : dict
        {'saline': {...}, 'dcz_random_subsampled': {...}} -- same shape as
        the sequential-block notebook's session_data (including
        'layer_of_cell'), so build_trial_matched_comparison_table/
        plot_response_plots_across_blocks work unchanged. The dcz entry's
        norm_spatial_activity is the FIRST repeat's trial subset only
        (illustrative example, not the aggregate -- see markdown above).
    """
    saline_info = session_catalog[saline_label]
    dcz_info = session_catalog[dcz_label]

    saline_activity, saline_norm, saline_bins = load_session_spatial_data(saline_info['tseries_dir'])
    dcz_activity, dcz_norm, dcz_bins = load_session_spatial_data(dcz_info['tseries_dir'])

    saline_layer_of_cell = load_session_layer_of_cell(saline_info['tseries_dir'], saline_activity.shape[0])
    dcz_layer_of_cell = load_session_layer_of_cell(dcz_info['tseries_dir'], dcz_activity.shape[0])

    n_trials_saline = saline_activity.shape[1]

    print(f"\n--- saline ({saline_label}) ---")
    saline_result = run_smi_and_reliability_for_trial_subset(
        saline_activity, saline_bins, np.arange(n_trials_saline), n_shuffles=n_shuffles
    )

    print(f"\n--- dcz ({dcz_label}), {n_repeats} random draw(s) of {n_trials_saline} trials each ---")
    repeat_results = repeated_random_subsample_smi(
        dcz_activity, dcz_bins, n_trials_saline, n_repeats=n_repeats, n_shuffles=n_shuffles
    )
    aggregated = aggregate_repeated_subsamples(repeat_results)

    dcz_result = {
        'n_trials': n_trials_saline,
        'n_repeats': aggregated['n_repeats'],
        'analysis_reliable_cells': aggregated['analysis_reliable_cells'],
        'valid_cells_mask': aggregated['valid_cells_mask'],
        'SMI': aggregated['median_SMI'],
        'mean_SMI': aggregated['mean_SMI'],
        'valid_fraction': aggregated['valid_fraction'],
        'reliable_fraction': aggregated['reliable_fraction'],
    }

    block_results = {'saline': saline_result, 'dcz_random_subsampled': dcz_result}

    first_repeat_indices = repeat_results[0]['trial_indices']
    session_data = {
        'saline': {
            'spatial_activity': saline_activity,
            'norm_spatial_activity': saline_norm,
            'bin_centers': saline_bins,
            'layer_of_cell': saline_layer_of_cell,
        },
        'dcz_random_subsampled': {
            'spatial_activity': dcz_activity[:, first_repeat_indices, :],
            'norm_spatial_activity': dcz_norm[:, first_repeat_indices, :],
            'bin_centers': dcz_bins,
            'layer_of_cell': dcz_layer_of_cell,
        },
    }

    return block_results, session_data

In [ ]:
# --- Try it on the real first pair, small n_repeats for speed ---
test_block_results, test_session_data = run_trial_matched_random_subsample_comparison(
    smi_catalog, test_pair['saline_label'], test_pair['dcz_label'], n_repeats=3
)
print(f"\nConditions: {list(test_block_results.keys())}")

## Function -- `run_trial_matched_random_subsample_comparison_v2`

Drop-in alternative to `run_trial_matched_random_subsample_comparison` above: same everything, except the dcz condition's `analysis_reliable_cells` comes from `aggregate_repeated_subsamples_by_averaged_evidence` instead of `aggregate_repeated_subsamples`'s majority vote. `median_SMI`/`mean_SMI` still come from `aggregate_repeated_subsamples` (SMI's own value/positional-validity computation doesn't change) -- only which cells count as reliable changes. `valid_cells_mask` = reliable AND has a usable (non-NaN) median SMI estimate, mirroring how `calculate_SMI_improved` itself ANDs reliability with positional validity.

Returns the exact same `block_results`/`session_data` shape as v1, so every downstream function (`build_trial_matched_comparison_table`, `plot_smi_saline_vs_subsampled`, `plot_response_plots_saline_vs_subsampled`, `save_random_subsample_outputs`) works unchanged on either version's output -- run both on the same pair and compare before deciding which aggregation to keep for the full across-all-groups driver at the bottom.</cell id="73e163c7">


In [ ]:
def run_trial_matched_random_subsample_comparison_v2(session_catalog, saline_label, dcz_label,
                                                       n_repeats=N_REPEATS, n_shuffles=N_SHUFFLES):
    """
    Same as run_trial_matched_random_subsample_comparison, but derives the
    dcz condition's analysis_reliable_cells from
    aggregate_repeated_subsamples_by_averaged_evidence (averaged continuous
    evidence, single threshold) instead of aggregate_repeated_subsamples's
    majority vote -- see that function's docstring for why.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    saline_label, dcz_label : str
    n_repeats, n_shuffles : int

    Returns
    -------
    block_results : dict
        {'saline': ..., 'dcz_random_subsampled': ...}.
    session_data : dict
        {'saline': {...}, 'dcz_random_subsampled': {...}}.
    """
    saline_info = session_catalog[saline_label]
    dcz_info = session_catalog[dcz_label]

    saline_activity, saline_norm, saline_bins = load_session_spatial_data(saline_info['tseries_dir'])
    dcz_activity, dcz_norm, dcz_bins = load_session_spatial_data(dcz_info['tseries_dir'])

    saline_layer_of_cell = load_session_layer_of_cell(saline_info['tseries_dir'], saline_activity.shape[0])
    dcz_layer_of_cell = load_session_layer_of_cell(dcz_info['tseries_dir'], dcz_activity.shape[0])

    n_trials_saline = saline_activity.shape[1]

    print(f"\n--- saline ({saline_label}) ---")
    saline_result = run_smi_and_reliability_for_trial_subset(
        saline_activity, saline_bins, np.arange(n_trials_saline), n_shuffles=n_shuffles
    )

    print(f"\n--- dcz ({dcz_label}), {n_repeats} random draw(s) of {n_trials_saline} trials each ---")
    repeat_results = repeated_random_subsample_smi(
        dcz_activity, dcz_bins, n_trials_saline, n_repeats=n_repeats, n_shuffles=n_shuffles
    )
    aggregated = aggregate_repeated_subsamples(repeat_results)
    aggregated_evidence = aggregate_repeated_subsamples_by_averaged_evidence(repeat_results)

    # Reliability comes from the averaged-evidence call; SMI value + which
    # cells have a usable (non-NaN) SMI estimate still come from
    # aggregate_repeated_subsamples (unaffected by which reliability call is used).
    analysis_reliable_cells = aggregated_evidence['analysis_reliable_cells']
    valid_cells_mask = analysis_reliable_cells & ~np.isnan(aggregated['median_SMI'])

    dcz_result = {
        'n_trials': n_trials_saline,
        'n_repeats': aggregated['n_repeats'],
        'analysis_reliable_cells': analysis_reliable_cells,
        'valid_cells_mask': valid_cells_mask,
        'SMI': aggregated['median_SMI'],
        'mean_SMI': aggregated['mean_SMI'],
        'active_fraction': aggregated_evidence['active_fraction'],
        'avg_cc_mean': aggregated_evidence['avg_cc_mean'],
        'cohen_d_mean': aggregated_evidence['cohen_d_mean'],
        'peak_distance_mean': aggregated_evidence['peak_distance_mean'],
    }

    block_results = {'saline': saline_result, 'dcz_random_subsampled': dcz_result}

    first_repeat_indices = repeat_results[0]['trial_indices']
    session_data = {
        'saline': {
            'spatial_activity': saline_activity,
            'norm_spatial_activity': saline_norm,
            'bin_centers': saline_bins,
            'layer_of_cell': saline_layer_of_cell,
        },
        'dcz_random_subsampled': {
            'spatial_activity': dcz_activity[:, first_repeat_indices, :],
            'norm_spatial_activity': dcz_norm[:, first_repeat_indices, :],
            'bin_centers': dcz_bins,
            'layer_of_cell': dcz_layer_of_cell,
        },
    }

    return block_results, session_data

## Function -- `compute_pooled_reliability_evidence` (v3 -- pools raw real/null correlations across repeats)

Real-data check on JSY090 DCZ_1 found `aggregate_repeated_subsamples_by_averaged_evidence` (v2, above) actually performs *worse* than the original vote (`aggregate_repeated_subsamples`): it recovered only 19/152 (12.5%) of the cells reliable on the full dcz session vs. the vote's 37/152 (24.3%), and every one of the 30 cells it called reliable that the vote didn't were **not** reliable on the full session -- straight-up false positives. The cause: v2 averaged `avg_cc` across repeats but compared it to a flat floor (`min_cc_threshold`) instead of each cell's own chance/null distribution -- the shuffle-relative comparison isn't a redundant extra check, it's what gives the test its specificity, and a flat floor doesn't survive noise-averaging the same way (many never-tuned cells' *averaged* correlation drifts above a flat floor once its noise is smoothed out, even though none of their individual repeats ever beat their own proper null).

This function fixes that by properly reproducing the null comparison at the pooled level instead of dropping it: for each repeat (reusing its exact `trial_indices` from `repeated_random_subsample_smi`, nothing redrawn), it recomputes the real odd/even correlation and a full `n_shuffles`-sized null distribution per cell -- mirroring `test_cell_reliability_improved`'s internal math exactly (even/odd split, circular-shift shuffle), but keeping the raw null array instead of collapsing it to a single threshold-crossing boolean the way that helper does internally. Pooling every repeat's real cc (`N_REPEATS` independent-trial-subset observations) and every repeat's null cc (`N_REPEATS x n_shuffles` observations) gives both a lower-variance real-correlation estimate *and* a much more precisely-estimated null distribution than any single repeat has -- so the shuffle-percentile/min-cc/Cohen's-d comparison happens once against properly pooled evidence, keeping the chance-correction the flat-floor version lost. Activity threshold (`improved_activity_threshold_check`) and the pattern test (`evaluate_pattern_similarity_improved`) are reused directly from `helper.ReliabilityTesting`, unmodified -- neither is shuffle-based, so neither was the specificity problem; only the correlation test's internal loop needed reimplementing here to expose what `combined_reliability_test_improved` discards.</cell id="e63ec7e7">


In [ ]:
def compute_pooled_reliability_evidence(dcz_activity, dcz_bins, repeat_results, n_shuffles=N_SHUFFLES,
                                         reliability_kwargs=RELIABILITY_KWARGS, smi_kwargs=SMI_KWARGS,
                                         activity_fraction_threshold=VALID_FRACTION_THRESHOLD):
    """
    Pools the raw correlation evidence across repeated_random_subsample_smi's
    repeats into ONE reliability decision -- see markdown above for why
    this replaces both aggregate_repeated_subsamples's vote and
    aggregate_repeated_subsamples_by_averaged_evidence's flat-floor
    average.

    Also applies the onset/reward-response exclusion (filter_onset_response_cells,
    reused unmodified) per repeat and requires a cell to be non-onset in
    >= activity_fraction_threshold of repeats -- an earlier version of this
    function omitted this (only reconstructing the correlation/pattern/
    cohen/activity criteria), and a real-data check found this let ~30
    onset/reward-artifact cells through as apparent "newly recovered"
    reliable cells: such cells are often highly REPRODUCIBLE (which is
    exactly why they pass correlation/cohen's-d/pattern tests easily) but
    aren't place-tuned -- which is why every per-repeat computation
    elsewhere in this pipeline ANDs combined_reliable with non_onset_cells,
    and this aggregate-level decision needs to as well.

    Parameters
    ----------
    dcz_activity : numpy.ndarray
        (n_cells, n_trials, n_bins) -- the dcz session's FULL array (the
        same one repeated_random_subsample_smi drew trial_indices from).
    dcz_bins : numpy.ndarray
        Raw cm-scale bin centers for the dcz session (for the onset filter's
        scaled_bin_centers, same rescaling run_smi_and_reliability_for_trial_subset uses).
    repeat_results : list of dict
        From repeated_random_subsample_smi -- only 'trial_indices' is
        reused from each entry (nothing is redrawn, so this stays exactly
        in sync with what SMI was already computed on).
    n_shuffles : int
    reliability_kwargs : dict
        Reuses cc_percentile/cohen_threshold/min_cc_threshold/
        min_pattern_corr/peak_distance_threshold/activity_method.
    smi_kwargs : dict
        Reuses exclude_first_bins/exclude_last_bins for the onset filter.
    activity_fraction_threshold : float
        Fraction of repeats a cell must be 'active' (and, separately,
        non-onset) in to count as such in the aggregate -- kept as a
        majority vote for both: neither the activity threshold nor a
        cell's onset/reward peak location is shuffle-based, so neither is
        the noisy part this function's pooling is meant to fix.

    Returns
    -------
    pooled : dict
        {'real_cc_mean', 'null_cc_threshold', 'cohen_d_pooled',
        'active_fraction', 'non_onset_fraction', 'pattern_corr_mean',
        'peak_distance_mean', 'analysis_reliable_cells'} -- each
        array-valued entry shape (n_cells,).
    """
    from helper.ReliabilityTesting import improved_activity_threshold_check, evaluate_pattern_similarity_improved

    n_repeats = len(repeat_results)
    n_cells = dcz_activity.shape[0]
    n_bins = dcz_activity.shape[2]

    shifted_centers = dcz_bins - np.min(dcz_bins)
    scaled_bin_centers = shifted_centers * (np.size(dcz_bins) / np.max(shifted_centers))

    real_cc_stack = np.full((n_repeats, n_cells), np.nan)
    null_cc_all = []  # n_repeats arrays of shape (n_cells, n_shuffles)
    active_stack = np.zeros((n_repeats, n_cells), dtype=bool)
    non_onset_stack = np.zeros((n_repeats, n_cells), dtype=bool)
    pattern_corr_stack = np.full((n_repeats, n_cells), np.nan)
    peak_dist_stack = np.full((n_repeats, n_cells), np.nan)

    for repeat_idx, repeat_result in enumerate(repeat_results):
        trial_indices = repeat_result['trial_indices']
        subset_activity = dcz_activity[:, trial_indices, :]
        n_trials = subset_activity.shape[1]

        active_cells, _ = improved_activity_threshold_check(
            subset_activity, method=reliability_kwargs['activity_method'])
        active_stack[repeat_idx] = active_cells

        non_onset_cells, _ = filter_onset_response_cells(
            subset_activity, scaled_bin_centers, reliable_cells=None,
            exclude_first_bins=smi_kwargs['exclude_first_bins'],
            exclude_last_bins=smi_kwargs['exclude_last_bins'],
            verbose=False,
        )
        non_onset_stack[repeat_idx] = non_onset_cells

        _, odd_even_corr, peak_distances = evaluate_pattern_similarity_improved(
            subset_activity, min_pattern_corr=reliability_kwargs['min_pattern_corr'],
            peak_distance_threshold=reliability_kwargs['peak_distance_threshold'])
        pattern_corr_stack[repeat_idx] = odd_even_corr
        peak_dist_stack[repeat_idx] = peak_distances

        trials1 = np.arange(0, n_trials, 2)
        trials2 = np.arange(1, n_trials, 2)
        null_cc_repeat = np.full((n_cells, n_shuffles), np.nan)

        print(f"  [pooled evidence] repeat {repeat_idx + 1}/{n_repeats}: "
              f"{int(active_cells.sum())}/{n_cells} active, {int(non_onset_cells.sum())}/{n_cells} non-onset")

        for cell in range(n_cells):
            if not active_cells[cell]:
                continue
            cell_activity = subset_activity[cell]
            first_half_mean = np.mean(cell_activity[trials1], axis=0)
            second_half_mean = np.mean(cell_activity[trials2], axis=0)
            cc = np.corrcoef(first_half_mean, second_half_mean)[0, 1]
            real_cc_stack[repeat_idx, cell] = cc if not np.isnan(cc) else 0

            for shuffle in range(n_shuffles):
                activity_rand = np.zeros_like(cell_activity)
                for trial in range(n_trials):
                    shift = np.random.randint(n_bins)
                    activity_rand[trial] = np.roll(cell_activity[trial], shift)
                first_half_rand_mean = np.mean(activity_rand[trials1], axis=0)
                second_half_rand_mean = np.mean(activity_rand[trials2], axis=0)
                cc_rand = np.corrcoef(first_half_rand_mean, second_half_rand_mean)[0, 1]
                null_cc_repeat[cell, shuffle] = cc_rand if not np.isnan(cc_rand) else 0

        null_cc_all.append(null_cc_repeat)

    active_fraction = active_stack.mean(axis=0)
    non_onset_fraction = non_onset_stack.mean(axis=0)
    pattern_corr_mean = np.nanmean(pattern_corr_stack, axis=0)
    peak_distance_mean = np.nanmean(peak_dist_stack, axis=0)

    real_cc_mean = np.full(n_cells, np.nan)
    null_cc_threshold = np.full(n_cells, np.nan)
    cohen_d_pooled = np.full(n_cells, np.nan)

    for cell in range(n_cells):
        active_mask = active_stack[:, cell]
        if not active_mask.any():
            continue
        real_vals = real_cc_stack[active_mask, cell]
        null_vals = np.concatenate([null_cc_all[r][cell] for r in range(n_repeats) if active_stack[r, cell]])

        real_cc_mean[cell] = real_vals.mean()
        null_cc_threshold[cell] = np.percentile(null_vals, reliability_kwargs['cc_percentile'])

        mean_diff = real_vals.mean() - null_vals.mean()
        n1, n2 = len(real_vals), len(null_vals)
        var1 = np.var(real_vals, ddof=1) if n1 > 1 else 0.0
        var2 = np.var(null_vals, ddof=1) if n2 > 1 else 0.0
        pooled_sd = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2)) if (n1 + n2) > 2 else 0.0
        cohen_d_pooled[cell] = mean_diff / pooled_sd if pooled_sd > 0 else 0.0

    active_in_aggregate = active_fraction >= activity_fraction_threshold
    non_onset_in_aggregate = non_onset_fraction >= activity_fraction_threshold
    reliable_by_cc = np.nan_to_num(real_cc_mean, nan=-np.inf) > reliability_kwargs['min_cc_threshold']
    reliable_by_shuffle = np.nan_to_num(real_cc_mean, nan=-np.inf) > np.nan_to_num(null_cc_threshold, nan=np.inf)
    reliable_by_cohen = np.nan_to_num(cohen_d_pooled, nan=-np.inf) > reliability_kwargs['cohen_threshold']
    pattern_reliable_agg = ((np.nan_to_num(pattern_corr_mean, nan=-np.inf) >= reliability_kwargs['min_pattern_corr'])
                            & (peak_distance_mean <= reliability_kwargs['peak_distance_threshold']))

    combined_reliable_agg = (active_in_aggregate & non_onset_in_aggregate & reliable_by_cc
                              & reliable_by_shuffle & reliable_by_cohen & pattern_reliable_agg)

    print(f"Pooled-evidence aggregate ({n_repeats} repeats x {n_shuffles} shuffles pooled): "
          f"{int(active_in_aggregate.sum())}/{n_cells} active, "
          f"{int(non_onset_in_aggregate.sum())}/{n_cells} non-onset, "
          f"{int(combined_reliable_agg.sum())}/{n_cells} reliable")

    return {
        'real_cc_mean': real_cc_mean,
        'null_cc_threshold': null_cc_threshold,
        'cohen_d_pooled': cohen_d_pooled,
        'active_fraction': active_fraction,
        'non_onset_fraction': non_onset_fraction,
        'pattern_corr_mean': pattern_corr_mean,
        'peak_distance_mean': peak_distance_mean,
        'analysis_reliable_cells': combined_reliable_agg,
    }


## Function -- `run_trial_matched_random_subsample_comparison_v3`

Drop-in alternative to v1/v2 above, using `compute_pooled_reliability_evidence` for the dcz condition's `analysis_reliable_cells` instead of either `aggregate_repeated_subsamples`'s vote or `aggregate_repeated_subsamples_by_averaged_evidence`'s flat-floor average. `median_SMI`/`mean_SMI` still come from `aggregate_repeated_subsamples` (unaffected). Returns the exact same `block_results`/`session_data` shape, so every downstream function works unchanged.</cell id="f6ebb29d">


In [ ]:
def run_trial_matched_random_subsample_comparison_v3(session_catalog, saline_label, dcz_label,
                                                       n_repeats=N_REPEATS, n_shuffles=N_SHUFFLES):
    """
    Same as run_trial_matched_random_subsample_comparison, but derives the
    dcz condition's analysis_reliable_cells from
    compute_pooled_reliability_evidence (pooled real/null correlations
    across repeats, single threshold) instead of aggregate_repeated_subsamples's
    majority vote or aggregate_repeated_subsamples_by_averaged_evidence's
    flat-floor average -- see that function's docstring for why.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    saline_label, dcz_label : str
    n_repeats, n_shuffles : int

    Returns
    -------
    block_results : dict
        {'saline': ..., 'dcz_random_subsampled': ...}.
    session_data : dict
        {'saline': {...}, 'dcz_random_subsampled': {...}}.
    """
    saline_info = session_catalog[saline_label]
    dcz_info = session_catalog[dcz_label]

    saline_activity, saline_norm, saline_bins = load_session_spatial_data(saline_info['tseries_dir'])
    dcz_activity, dcz_norm, dcz_bins = load_session_spatial_data(dcz_info['tseries_dir'])

    saline_layer_of_cell = load_session_layer_of_cell(saline_info['tseries_dir'], saline_activity.shape[0])
    dcz_layer_of_cell = load_session_layer_of_cell(dcz_info['tseries_dir'], dcz_activity.shape[0])

    n_trials_saline = saline_activity.shape[1]

    print(f"\n--- saline ({saline_label}) ---")
    saline_result = run_smi_and_reliability_for_trial_subset(
        saline_activity, saline_bins, np.arange(n_trials_saline), n_shuffles=n_shuffles
    )

    print(f"\n--- dcz ({dcz_label}), {n_repeats} random draw(s) of {n_trials_saline} trials each ---")
    repeat_results = repeated_random_subsample_smi(
        dcz_activity, dcz_bins, n_trials_saline, n_repeats=n_repeats, n_shuffles=n_shuffles
    )
    aggregated = aggregate_repeated_subsamples(repeat_results)
    pooled = compute_pooled_reliability_evidence(dcz_activity, dcz_bins, repeat_results, n_shuffles=n_shuffles)

    # Reliability comes from the pooled-evidence call; SMI value + which
    # cells have a usable (non-NaN) SMI estimate still come from
    # aggregate_repeated_subsamples (unaffected by which reliability call is used).
    analysis_reliable_cells = pooled['analysis_reliable_cells']
    valid_cells_mask = analysis_reliable_cells & ~np.isnan(aggregated['median_SMI'])

    dcz_result = {
        'n_trials': n_trials_saline,
        'n_repeats': aggregated['n_repeats'],
        'analysis_reliable_cells': analysis_reliable_cells,
        'valid_cells_mask': valid_cells_mask,
        'SMI': aggregated['median_SMI'],
        'mean_SMI': aggregated['mean_SMI'],
        'active_fraction': pooled['active_fraction'],
        'real_cc_mean': pooled['real_cc_mean'],
        'null_cc_threshold': pooled['null_cc_threshold'],
        'cohen_d_pooled': pooled['cohen_d_pooled'],
        'peak_distance_mean': pooled['peak_distance_mean'],
    }

    block_results = {'saline': saline_result, 'dcz_random_subsampled': dcz_result}

    first_repeat_indices = repeat_results[0]['trial_indices']
    session_data = {
        'saline': {
            'spatial_activity': saline_activity,
            'norm_spatial_activity': saline_norm,
            'bin_centers': saline_bins,
            'layer_of_cell': saline_layer_of_cell,
        },
        'dcz_random_subsampled': {
            'spatial_activity': dcz_activity[:, first_repeat_indices, :],
            'norm_spatial_activity': dcz_norm[:, first_repeat_indices, :],
            'bin_centers': dcz_bins,
            'layer_of_cell': dcz_layer_of_cell,
        },
    }

    return block_results, session_data

## Function -- `build_trial_matched_comparison_table`

Reused unchanged from `4.SessionComparison_TrialMatched.ipynb` -- same per-cell-row shape (`cell_idx, SMI, valid, analysis_reliable, layer, condition`), so a Phase-5-style layer analysis can consume this notebook's output too, the same way it consumes the sequential-block notebook's.

In [10]:
def build_trial_matched_comparison_table(block_results, session_data):
    """
    Build a per-cell-row table -- cell_idx, SMI, valid, analysis_reliable,
    layer, condition -- one row per cell per condition.

    Parameters
    ----------
    block_results : dict
        From run_trial_matched_random_subsample_comparison.
    session_data : dict
        From run_trial_matched_random_subsample_comparison (needs
        'layer_of_cell' per condition).

    Returns
    -------
    df : pandas.DataFrame
    """
    condition_dfs = []
    for condition_name, result in block_results.items():
        n_cells = len(result['SMI'])
        layer_of_cell = session_data[condition_name]['layer_of_cell']
        condition_dfs.append(pd.DataFrame({
            'cell_idx': np.arange(n_cells),
            'SMI': result['SMI'],
            'valid': result['valid_cells_mask'],
            'analysis_reliable': result['analysis_reliable_cells'],
            'layer': layer_of_cell,
            'condition': condition_name,
        }))

    df = pd.concat(condition_dfs, ignore_index=True)
    print(f"Built comparison table: {len(df)} cell-rows across {len(block_results)} condition(s)")
    return df

In [ ]:
# --- Try it on the real test results ---
test_comparison_table = build_trial_matched_comparison_table(test_block_results, test_session_data)
print(test_comparison_table.groupby('condition')['valid'].agg(['sum', 'count']))

## Function -- `plot_smi_saline_vs_subsampled`

Single violin+strip plot, saline vs. the aggregated dcz -- same visual style as `4.SessionComparison_TrialMatched.ipynb`'s `plot_smi_across_blocks` (in fact the exact same color/ordering logic works unchanged here too, since it only assumes `'saline'` is the first condition and colors everything else via a Purples gradient -- no `dcz_block_N`-specific string parsing).

In [11]:
def plot_smi_saline_vs_subsampled(block_results, title=''):
    """
    Single violin+strip plot of SMI (valid_cells_mask-filtered), saline
    vs. the aggregated random-subsampled dcz.

    Parameters
    ----------
    block_results : dict
        From run_trial_matched_random_subsample_comparison.
    title : str

    Returns
    -------
    fig : matplotlib.figure.Figure
    summary_df : pandas.DataFrame
        One row per condition: n, median_SMI, mean_SMI.
    """
    condition_order = list(block_results.keys())

    n_other = len(condition_order) - 1
    other_colors = plt.cm.Purples(np.linspace(0.4, 0.9, max(n_other, 1)))
    color_map = {'saline': 'tab:orange'}
    for i, cond in enumerate(condition_order[1:]):
        color_map[cond] = other_colors[i]

    data_by_condition = []
    summary_rows = []
    for cond in condition_order:
        result = block_results[cond]
        vals = result['SMI'][result['valid_cells_mask']]
        data_by_condition.append(vals)
        summary_rows.append({
            'condition': cond,
            'n': len(vals),
            'median_SMI': float(np.median(vals)) if len(vals) else np.nan,
            'mean_SMI': float(np.mean(vals)) if len(vals) else np.nan,
        })
    summary_df = pd.DataFrame(summary_rows)

    fig, ax = plt.subplots(figsize=(2.5 * len(condition_order) + 2, 8))
    nonempty = [(i, d) for i, d in enumerate(data_by_condition) if len(d) > 0]
    if nonempty:
        parts = ax.violinplot([d for _, d in nonempty], positions=[i + 1 for i, _ in nonempty],
                               showmedians=True)
        for (i, _), body in zip(nonempty, parts['bodies']):
            body.set_facecolor(color_map[condition_order[i]])
            body.set_alpha(0.4)

    rng = np.random.default_rng(0)
    for i, vals in enumerate(data_by_condition):
        if len(vals) == 0:
            continue
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                   color=color_map[condition_order[i]], s=15, alpha=0.5)

    ax.set_xticks(range(1, len(condition_order) + 1))
    ax.set_xticklabels([f"{c}\n(n={r['n']})" for c, r in zip(condition_order, summary_rows)])
    ax.set_ylabel('SMI (median across repeats where valid)')
    ax.set_title(title)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    return fig, summary_df

In [ ]:
# --- Try it on the real test results ---
test_smi_fig, test_summary_df = plot_smi_saline_vs_subsampled(
    test_block_results, title=f"{test_pair['group_name']} -- saline vs. random-subsampled dcz"
)
print(test_summary_df.to_string(index=False))
# Not calling plt.show() here -- see the "no popup windows" note further down.
# plt.show()

## Function -- `plot_response_plots_saline_vs_subsampled`

Same three-tier (`analysis_reliable_cells`/`valid_cells_mask`/rejected-from-valid) `create_response_plot` diagnostic as the sequential-block notebook, reused unchanged -- but remember the dcz panel here shows only the *first* repeat's trial subset (an illustrative example), not the full aggregate.

In [12]:
def plot_response_plots_saline_vs_subsampled(block_results, session_data):
    """
    create_response_plot for saline and the (first-repeat-illustrated) dcz
    condition -- three mask tiers each.

    Parameters
    ----------
    block_results : dict
        From run_trial_matched_random_subsample_comparison.
    session_data : dict
        From run_trial_matched_random_subsample_comparison.

    Returns
    -------
    figs : dict
        {(mask_name, condition_name): fig}.
    """
    mask_tiers = ['analysis_reliable_cells', 'valid_cells_mask', 'rejected_from_valid']
    figs = {}
    for condition_name, result in block_results.items():
        norm_activity = session_data[condition_name]['norm_spatial_activity']
        masks = {
            'analysis_reliable_cells': result['analysis_reliable_cells'],
            'valid_cells_mask': result['valid_cells_mask'],
            'rejected_from_valid': result['analysis_reliable_cells'] & ~result['valid_cells_mask'],
        }
        for mask_name in mask_tiers:
            mask = masks[mask_name]
            if not mask.any():
                print(f"  {condition_name} / {mask_name}: 0 cells -- skipping (nothing to plot).")
                continue
            fig, _ = create_response_plot(norm_activity, mask, clim=(0, 1))
            label_suffix = ' (1st draw, illustrative)' if condition_name == 'dcz_random_subsampled' else ''
            fig.suptitle(f"{condition_name}{label_suffix}\n{mask_name} (n={int(mask.sum())})", fontsize=14)
            figs[(mask_name, condition_name)] = fig

    return figs

In [ ]:
# --- Try it on the real test results ---
test_response_figs = plot_response_plots_saline_vs_subsampled(test_block_results, test_session_data)
print(f"Got {len(test_response_figs)} figure(s).")

## Functions -- save everything + run it across every saline/dcz pair, for every animal

Saved under a **different filename/output directory** than the sequential-block notebook (`Phase4_TrialMatched_RandomSubsample_Results`, `*_random_subsampled_comparison_table.csv`) so the two methods' results never collide. Same no-popup-window, dual-animal, per-pair-error-handling conventions as `4.SessionComparison_TrialMatched.ipynb`.

In [13]:
def save_random_subsample_outputs(output_dir, group_name, summary_df, smi_fig, response_figs, comparison_table):
    """
    Save one group's outputs: SMI summary table, SMI comparison figure,
    every response-plot figure, and the per-cell comparison table.

    Parameters
    ----------
    output_dir : str
    group_name : str
    summary_df : pandas.DataFrame
        From plot_smi_saline_vs_subsampled.
    smi_fig : matplotlib.figure.Figure
        From plot_smi_saline_vs_subsampled.
    response_figs : dict
        From plot_response_plots_saline_vs_subsampled.
    comparison_table : pandas.DataFrame
        From build_trial_matched_comparison_table.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {
        'summary': save_dataframe_csv(summary_df, output_dir, f"{group_name}_random_subsampled_smi_summary.csv"),
        'smi_plot': save_figure_png(smi_fig, output_dir, f"{group_name}_random_subsampled_smi_comparison.png"),
        'comparison_table': save_dataframe_csv(
            comparison_table, output_dir, f"{group_name}_random_subsampled_comparison_table.csv"),
    }
    plt.close(smi_fig)

    response_dir = os.path.join(output_dir, f"{group_name}_response_plots")
    for (mask_name, condition_name), fig in response_figs.items():
        key = f"{condition_name}_{mask_name}"
        saved_paths[key] = save_figure_png(fig, response_dir, f"{key}.png")
        plt.close(fig)

    return saved_paths


def run_all_random_subsample_comparisons(session_catalog, pairs, output_dir, n_repeats=N_REPEATS, n_shuffles=N_SHUFFLES):
    """
    Loop the whole random-subsample pipeline over every saline/dcz pair.

    Parameters
    ----------
    session_catalog : dict
    pairs : list of dict
        From pair_saline_dcz_sessions.
    output_dir : str
    n_repeats, n_shuffles : int

    Returns
    -------
    results_by_group : dict
        {group_name: {'block_results', 'session_data', 'summary_df',
        'comparison_table', 'saved_paths'}}.
    """
    results_by_group = {}
    failed_groups = {}
    for pair in pairs:
        group_name = pair['group_name']
        print(f"\n{'='*90}\nRANDOM-SUBSAMPLE -- {group_name}\n{'='*90}")

        try:
            block_results, session_data = run_trial_matched_random_subsample_comparison(
                session_catalog, pair['saline_label'], pair['dcz_label'],
                n_repeats=n_repeats, n_shuffles=n_shuffles
            )

            smi_fig, summary_df = plot_smi_saline_vs_subsampled(
                block_results, title=f"{group_name} -- saline vs. random-subsampled dcz")
            print(summary_df.to_string(index=False))

            response_figs = plot_response_plots_saline_vs_subsampled(block_results, session_data)
            comparison_table = build_trial_matched_comparison_table(block_results, session_data)

            saved_paths = save_random_subsample_outputs(
                output_dir, group_name, summary_df, smi_fig, response_figs, comparison_table)

            results_by_group[group_name] = {
                'block_results': block_results,
                'session_data': session_data,
                'summary_df': summary_df,
                'comparison_table': comparison_table,
                'saved_paths': saved_paths,
            }
        except Exception as exc:
            print(f"FAILED on group '{group_name}' -- {type(exc).__name__}: {exc}")
            print(f"  Skipping this group and continuing with the rest. "
                  f"(Any groups already completed above are already saved to disk.)")
            failed_groups[group_name] = exc
            continue

    if failed_groups:
        print(f"\n{len(failed_groups)}/{len(pairs)} group(s) failed and were skipped: {list(failed_groups.keys())}")

    return results_by_group

In [14]:
# --- Run it across every saline/dcz pair, for both animals ---
ANIMAL_CONFIGS = [
    {'animal_label': 'JSY090', 'animal_dir': TEST_ANIMAL_DIR_JSY090},
    {'animal_label': 'JSY093', 'animal_dir': TEST_ANIMAL_DIR_JSY093},
]

random_subsample_results_by_animal = {}
for animal_cfg in ANIMAL_CONFIGS:
    print(f"\n{'#'*90}\n{animal_cfg['animal_label']}\n{'#'*90}")
    animal_catalog = discover_smi_sessions(animal_cfg['animal_dir'])
    animal_pairs = pair_saline_dcz_sessions(animal_catalog)
    animal_output_dir = os.path.join(animal_cfg['animal_dir'], 'Phase4_TrialMatched_RandomSubsample_Results')
    random_subsample_results_by_animal[animal_cfg['animal_label']] = run_all_random_subsample_comparisons(
        animal_catalog, animal_pairs, animal_output_dir
    )


##########################################################################################
JSY090
##########################################################################################
Discovered 15 sessions with saved SMI results under D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD:
  [baseline] Day1__TSeries-07192026-0941-001  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260719_JSY_JSY090_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\Day1_smi_results_dreadd.h5
  [baseline] Day2__TSeries-07202026-1009-001  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260720_JSY_JSY090_LongitudinalImaging_DREADD_Day2\TSeries-07202026-1009-001\Day2_smi_results_dreadd.h5
  [baseline] Day3__TSeries-07212026-0907-001  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260721_JSY_JSY090_LongitudinalImaging_DREADD_Day3\TSeries-07212026-0907-001\Day3_smi_results_dreadd.h5
  [baseline] Day4__TSeries-07222026-18

Testing cell reliability: 100%|██████████| 578/578 [00:31<00:00, 18.56it/s]



Pattern Similarity Results:
  Mean odd-even correlation: 0.238
  Mean peak distance: 28.7 bins
  Cells with good correlation (>0.3): 141
  Cells with stable peaks (<5 bins): 222

Reliability Test Results:
  Active cells: 520
  Reliable cells (correlation test): 292
  Pattern consistent cells: 105
  Combined reliable cells: 87

Onset/Reward Filtering: Onset < 10.1cm, Reward > 119.9cm
  Rejected: 41 (Onset: 28, Reward: 13, Zero: 0)
  Corridor: 0.0 to 130.0 cm (length: 130.0 cm)
  Boundary exclusions: START 15.0 cm, END 10.0 cm
  Allowed region: 15.0 to 120.0 cm
  Allowed length: 105.0 cm
  Allowed bins: 105 out of 130


100%|██████████| 578/578 [00:13<00:00, 44.14it/s]



SMI calculation summary:
  Total cells: 578
  Valid cells: 559 (96.7%)
  Reliable & Valid cells: 46 (8.0%)
  Rejected - no peak in allowed region: 19
  Rejected - no valid non-preferred positions: 0
  Rejected - zero response sum: 0
  Rejected - boundary violations: 0
  Fitting failed (used raw values): 0
  20 trials: combined_reliable=87, analysis_reliable=46, valid=46

--- dcz (260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ), 20 random draw(s) of 20 trials each ---
--- Repeat 1/20 ---
Activity threshold (absolute_percentile): 0.0725
Cells passing activity threshold: 614/683 (89.9%)


...


SMI calculation summary:
  Total cells: 869
  Valid cells: 864 (99.4%)
  Reliable & Valid cells: 23 (2.6%)
  Rejected - no peak in allowed region: 5
  Rejected - no valid non-preferred positions: 0
  Rejected - zero response sum: 0
  Rejected - boundary violations: 0
  Fitting failed (used raw values): 0
  27 trials: combined_reliable=63, analysis_reliable=23, valid=23
--- Repeat 20/20 ---
Activity threshold (absolute_percentile): 0.0684
Cells passing activity threshold: 782/869 (90.0%)


Testing cell reliability: 100%|██████████| 869/869 [00:56<00:00, 15.32it/s]



Pattern Similarity Results:
  Mean odd-even correlation: 0.168
  Mean peak distance: 30.8 bins
  Cells with good correlation (>0.3): 113
  Cells with stable peaks (<5 bins): 232

Reliability Test Results:
  Active cells: 782
  Reliable cells (correlation test): 381
  Pattern consistent cells: 75
  Combined reliable cells: 66

Onset/Reward Filtering: Onset < 10.1cm, Reward > 119.9cm
  Rejected: 36 (Onset: 34, Reward: 2, Zero: 0)
  Corridor: 0.0 to 130.0 cm (length: 130.0 cm)
  Boundary exclusions: START 15.0 cm, END 10.0 cm
  Allowed region: 15.0 to 120.0 cm
  Allowed length: 105.0 cm
  Allowed bins: 105 out of 130


100%|██████████| 869/869 [00:23<00:00, 37.14it/s]



SMI calculation summary:
  Total cells: 869
  Valid cells: 864 (99.4%)
  Reliable & Valid cells: 30 (3.5%)
  Rejected - no peak in allowed region: 5
  Rejected - no valid non-preferred positions: 0
  Rejected - zero response sum: 0
  Rejected - boundary violations: 0
  Fitting failed (used raw values): 0
  27 trials: combined_reliable=66, analysis_reliable=30, valid=30
Aggregated 20 repeats: 12/869 cells valid in >=50% of repeats, 12/869 reliable in >=50%
            condition  n  median_SMI  mean_SMI
               saline 14    0.695430  0.650187
dcz_random_subsampled 12    0.690298  0.652464
  saline / rejected_from_valid: 0 cells -- skipping (nothing to plot).
  dcz_random_subsampled / rejected_from_valid: 0 cells -- skipping (nothing to plot).
Built comparison table: 1344 cell-rows across 2 condition(s)
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_TrialMatched_RandomSubsample_Results\260801_JSY_JSY093_LongitudinalImaging_DREADD_StationaryOpenLoo